In [1]:
# Import các thư viện cơ bản
import pandas as pd
import os
import joblib

# Import các công cụ từ scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Đọc dữ liệu
duong_dan_file = "../data/clean_data.csv"
df = pd.read_csv(duong_dan_file)

# Xóa các dòng bị lỗi trống (nếu có trong quá trình lưu file)
df = df.dropna(subset=['Cleaned_Review', 'Label'])

print("Tổng số dữ liệu sẵn sàng huấn luyện:", len(df))
df.head()

Tổng số dữ liệu sẵn sàng huấn luyện: 10


,Review,Cleaned_Review,Label
0,"Sản phẩm quá tệ, giao hàng chậm rì",quá tệ giao chậm rì,0
1,áo đẹp lắm nha shop iu <3,đẹp lắm yêu positive_icon,1
2,"chất vải nóng, mạc 1 lần là mún vứt r",chất_lượng vải nóng mặc muốn vứt,0
3,"đóng gói cẩn thận, 10 điểmmmm",đóng_gói cẩn_thận điểm,1
4,"hàng dỏm, mn đừng mua nha",kém đừng mua,0


In [2]:
from sklearn.model_selection import train_test_split

# Lấy tập văn bản làm đầu vào (X) và tập nhãn làm mục tiêu (y)
X = df['Cleaned_Review']
y = df['Label']

# Kiểm tra số lượng dữ liệu để quyết định cách chia
if len(X) < 50:
    print("[Cảnh báo] Dữ liệu hiện tại quá ít. Hệ thống sẽ chia ngẫu nhiên bình thường (Bỏ qua Stratify).")
    # Lần 1: Chia 70% Train, 30% Temp (Không dùng stratify)
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
    # Lần 2: Chia đôi Temp thành 15% Val và 15% Test
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

else:
    print("[OK] Dữ liệu đủ lớn. Đang tiến hành chia phân tầng (Stratify) theo chuẩn...")
    # Lần 1: Chia có stratify
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    # Lần 2: Chia có stratify
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("-" * 40)
print("Tổng số câu dữ liệu hiện có:", len(X))
print(f"Số câu tập Train (70%): {len(X_train)} câu")
print(f"Số câu tập Validation (15%): {len(X_val)} câu")
print(f"Số câu tập Test (15%): {len(X_test)} câu")

[Cảnh báo] Dữ liệu hiện tại quá ít. Hệ thống sẽ chia ngẫu nhiên bình thường (Bỏ qua Stratify).
----------------------------------------
Tổng số câu dữ liệu hiện có: 10
Số câu tập Train (70%): 7 câu
Số câu tập Validation (15%): 1 câu
Số câu tập Test (15%): 2 câu


In [3]:
# Khởi tạo công cụ TF-IDF
vectorizer = TfidfVectorizer()

# 1. Học từ vựng và biến đổi tập Train (Dùng fit_transform)
X_train_vector = vectorizer.fit_transform(X_train)

# 2. CHỈ BIẾN ĐỔI tập Val và tập Test (Chỉ dùng transform, tuyệt đối không dùng fit)
X_val_vector = vectorizer.transform(X_val)
X_test_vector = vectorizer.transform(X_test)

print("Kích thước ma trận Train:", X_train_vector.shape)
print("Kích thước ma trận Validation:", X_val_vector.shape)

Kích thước ma trận Train: (7, 31)
Kích thước ma trận Validation: (1, 31)


In [4]:
import os
import joblib
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("ĐANG TỰ ĐỘNG HUẤN LUYỆN VÀ TÌM KIẾM MÔ HÌNH TỐT NHẤT...\n")

# 1. Huấn luyện Naive Bayes
mo_hinh_nb = MultinomialNB()
mo_hinh_nb.fit(X_train_vector, y_train)
du_doan_nb = mo_hinh_nb.predict(X_val_vector)
# Dùng average='macro' để F1-score chấm điểm công bằng cho cả lớp Tích cực và Tiêu cực
f1_nb = f1_score(y_val, du_doan_nb, average='macro', zero_division=0) 

# 2. Huấn luyện Logistic Regression
mo_hinh_lr = LogisticRegression()
mo_hinh_lr.fit(X_train_vector, y_train)
du_doan_lr = mo_hinh_lr.predict(X_val_vector)
f1_lr = f1_score(y_val, du_doan_lr, average='macro', zero_division=0)

print(f"[Naive Bayes]        Điểm F1-Score: {round(f1_nb * 100, 2)}%")
print(f"[Logistic Reg]      Điểm F1-Score: {round(f1_lr * 100, 2)}%\n")

# 3. TỰ ĐỘNG SO SÁNH VÀ LỰA CHỌN (AI TỰ QUYẾT ĐỊNH)
if f1_lr > f1_nb:
    print("=> TỰ ĐỘNG CHỌN: LOGISTIC REGRESSION làm mô hình chính thức!")
    best_model = mo_hinh_lr
    best_predict = du_doan_lr
    ten_mo_hinh = "Logistic Regression"
else:
    # Nếu điểm bằng nhau hoặc NB cao hơn, chọn NB
    print("=> TỰ ĐỘNG CHỌN: NAIVE BAYES làm mô hình chính thức!")
    best_model = mo_hinh_nb
    best_predict = du_doan_nb
    ten_mo_hinh = "Naive Bayes"

# 4. In báo cáo chi tiết cho mô hình chiến thắng
print("-" * 55)
print(f"BẢNG BÁO CÁO CHI TIẾT CỦA THUẬT TOÁN {ten_mo_hinh.upper()}:")
# zero_division=0 giúp triệt tiêu các cảnh báo lỗi đỏ lòm khi dữ liệu quá ít
print(classification_report(y_val, best_predict, target_names=['Tiêu cực (0)', 'Tích cực (1)'], zero_division=0))
print("-" * 55)

# 5. Xuất file vật lý cho mô hình tốt nhất
thu_muc_luu = "../saved_models"
if not os.path.exists(thu_muc_luu):
    os.makedirs(thu_muc_luu)

# Lưu Vectorizer (Từ điển)
joblib.dump(vectorizer, os.path.join(thu_muc_luu, "tfidf_vectorizer.pkl"))

# Lưu Mô hình tốt nhất vừa được chọn ở bước 3
joblib.dump(best_model, os.path.join(thu_muc_luu, "sentiment_model.pkl"))

print(f"[+] Hoàn tất! Hệ thống đã lưu bộ não của {ten_mo_hinh} vào thư mục saved_models.")
# print("[+] Sẵn sàng bàn giao 2 file .pkl cho bước làm Giao diện Web!")

ĐANG TỰ ĐỘNG HUẤN LUYỆN VÀ TÌM KIẾM MÔ HÌNH TỐT NHẤT...

[Naive Bayes]        Điểm F1-Score: 0.0%
[Logistic Reg]      Điểm F1-Score: 0.0%

=> TỰ ĐỘNG CHỌN: NAIVE BAYES làm mô hình chính thức!
-------------------------------------------------------
BẢNG BÁO CÁO CHI TIẾT CỦA THUẬT TOÁN NAIVE BAYES:
              precision    recall  f1-score   support

Tiêu cực (0)       0.00      0.00      0.00       0.0
Tích cực (1)       0.00      0.00      0.00       1.0

    accuracy                           0.00       1.0
   macro avg       0.00      0.00      0.00       1.0
weighted avg       0.00      0.00      0.00       1.0

-------------------------------------------------------
[+] Hoàn tất! Hệ thống đã lưu bộ não của Naive Bayes vào thư mục saved_models.
